In [1]:
# ==========================================
# 1. Standard Library Imports
# ==========================================
import os
import sys
import csv
import time
import pickle
import random
import warnings
import itertools
import textwrap
import re
from collections import Counter

# ==========================================
# 2. Third-Party Scientific Imports
# ==========================================
import numpy as np
import pandas as pd
import xarray as xr
import scipy.stats as stats
import scipy.linalg as linalg
import scipy.optimize as optimize
import scipy.fft as fft
import scipy.signal as signal
from scipy.interpolate import interp1d
from scipy.stats import pearsonr, skewnorm

# Specialized Science Toolkits
import pywt
import pycwt as wavelet
import emd
from eofs.standard import Eof
import shap
import lime
from lime.lime_tabular import LimeTabularExplainer

# Machine Learning (Scikit-Learn)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, PolynomialFeatures, normalize
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedShuffleSplit, 
    cross_val_score, KFold, LeaveOneOut, StratifiedKFold
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, VotingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score, 
    roc_curve, accuracy_score, classification_report
)
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn import tree

# ==========================================
# 3. Visualization & Plotting
# ==========================================
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as patches
import matplotlib.patheffects as PathEffects
import seaborn as sns
import imgkit
from matplotlib.ticker import ScalarFormatter
from matplotlib.colors import BoundaryNorm

# Geospatial Plotting
import cartopy
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.util import add_cyclic_point

# ==========================================
# 4. Configuration & Local Modules
# ==========================================
warnings.filterwarnings('ignore')
sns.set_theme() # Optional: sets a cleaner default style for plots

# Define Paths
HURDAT_CODE_PATH = '/Users/jackskari/Desktop/NCSU_Research/Modularized_Code/'
DATA_FILE_PATH = "/Users/jackskari/Desktop/NCSU_Research/New_Data_2025/hurr2025.csv"

# Dynamic import of local module
if HURDAT_CODE_PATH not in sys.path:
    sys.path.append(HURDAT_CODE_PATH)

try:
    import ReadHurdatData
except ImportError:
    print(f"Error: Could not find ReadHurdatData at {HURDAT_CODE_PATH}")

# ==========================================
# 5. Data Loading & Initial Processing
# ==========================================
df = pd.read_csv(DATA_FILE_PATH)

# Load Near-Land Data
(nearmod_df, nearseasonal_ace, nearseasonal_num, nearseasonal_dur, 
 nearseasonal_apsm, nearseasonal_ap6h, nearseasonal_avdur, nearmonthly_ace, 
 nearseasonal_avmaxwind, nearseasonal_hurr, nearseasonal_mh, nearseasonal_hu, 
 nearseasonal_mjhu, yearz) = ReadHurdatData.hurdatclean(df, 1, 12, landmask=1, nearland=2)

# Load Standard Seasonal Data
(mod_df, seasonal_ace, seasonal_num, seasonal_dur, 
 seasonal_apsm, seasonal_ap6h, seasonal_avdur, monthly_ace, 
 seasonal_avmaxwind, seasonal_hurr, seasonal_mh, seasonal_hu, 
 seasonal_mjhu, yearz) = ReadHurdatData.hurdatclean(df, 1, 12)

# Slicing Indices (100+)
yearzz = np.asarray(yearz[100:])
ace_prime = seasonal_ace[100:]
num_prime = seasonal_num[100:]

nearace_prime = nearseasonal_ace[100:]
nearnum_prime = nearseasonal_num[100:]

# Load in Data

In [69]:
# ==========================================
# 1. Categorization Functions
# ==========================================

def get_activity_categories(data_list, reference_list, year_labels):
    """
    Categorizes years into High, Mid, or Low activity based on quartiles.
    
    Args:
        data_list: The values to categorize.
        reference_list: The baseline values used to calculate percentiles (e.g., a training period).
        year_labels: List of years corresponding to the data_list.
    """
    q1 = np.percentile(reference_list, 25)
    q3 = np.percentile(reference_list, 75)

    # Use NumPy select for fast, readable conditional mapping
    conditions = [
        (data_list <= q1),
        (data_list >= q3)
    ]
    scores = [0, 2] # 0: Low, 2: High
    
    # Default is 1 (Mid)
    final_scores = np.select(conditions, scores, default=1)
    
    # Filter years into separate lists for research analysis
    low  = [year_labels[i] for i, s in enumerate(final_scores) if s == 0]
    mid  = [year_labels[i] for i, s in enumerate(final_scores) if s == 1]
    high = [year_labels[i] for i, s in enumerate(final_scores) if s == 2]

    return high, mid, low, final_scores


def get_extended_categories(data_list, reference_list, year_labels, bound=2.0):
    """
    Categorizes years into Hyperactive, High, Mid, or Low activity.
    """
    q1 = np.percentile(reference_list, 33)
    q3 = np.percentile(reference_list, 67)
    high_threshold = bound * np.median(reference_list)

    conditions = [
        (data_list < q1),                               # Low
        (data_list >= q3) & (data_list <= high_threshold), # High
        (data_list > high_threshold)                    # Very High (Hyperactive)
    ]
    scores = [0, 2, 3] # 0:Low, 2:High, 3:Very High
    
    final_scores = np.select(conditions, scores, default=1) # Default 1: Mid

    # Organize years
    low  = [year_labels[i] for i, s in enumerate(final_scores) if s == 0]
    mid  = [year_labels[i] for i, s in enumerate(final_scores) if s == 1]
    high = [year_labels[i] for i, s in enumerate(final_scores) if s == 2]
    v_high = [year_labels[i] for i, s in enumerate(final_scores) if s == 3]

    return v_high, high, mid, low, final_scores

# ==========================================
# 2. Path Configuration & Data Loading
# ==========================================

BASE_DIR = "/Users/jackskari/Desktop/NCSU_Research/ML_Data_Classes"

# Load Pickled Data
with open(f"{BASE_DIR}/truedata1950-2025.pkl", 'rb') as f:
    pure_data = pickle.load(f)

# Load DataFrames and clean "Unnamed" columns immediately


ace_df = pd.read_csv(f"{BASE_DIR}/clusteredace.csv", index_col=0)
coast_ace_df = pd.read_csv(f"{BASE_DIR}/clusteredcoastace.csv", index_col=0)

with open(f"{BASE_DIR}/cpcclass.pkl", 'rb') as f:
    activity_class = pickle.load(f)


# ==========================================
# 3. Variable Initialization
# ==========================================

# Feature Names
feature_names = ['amm', 'amo', 'ao', 'censo', 'dm', 'epo', 'ggst', 'ngst', 'sgst',
    'mdrolr', 'mdrslp', 'mdrsst', 'nao', 'pdo', 'pna', 'qbo', 'sfi', 'soi', 'tni', 
    'tna', 'tsa', 'whwp', 'wpi', 'nino12', 'nino3', 'nino34', 'nino4', 'mei', 'roni', 
    'sahel', 'mdru200', 'mdru850', 'mdrv200', 'mdrv850', 'mdrvws', 'gomu200', 'gomu850',
    'gomv200', 'gomv850', 'gomvws', 'mdrt200', 'gomolr', 'gomslp', 'rhsal', 'nino34_forecast'
]

# Extract activity class components
(hyper_years, above_years, average_years, below_years, 
 score_data, actualscore) = activity_class[:6]

activity_bins = [below_years, average_years, above_years, hyper_years]
type_labels = ["Below Average", "Average", "Above Average", "Hyperactive"]

# ==========================================
# 4. Scenario Logic (Coastal vs General)
# ==========================================

COAST_MODE = 0
features = ace_df
class_type = 'CPC'

if COAST_MODE == 1:
    vhigh, high, mid, low, scores = get_extended_categories(
        nearace_prime, 
        nearace_prime[:70], 
        year_labels=yearzz, 
        bound=2
    )
    features = coast_ace_df
    actualscore = np.asarray(scores)
    class_type = 'CHACE'
# Metadata for plotting/exports
months_order = ['JUN','JUL','AUG','SEP','OCT','NOV','DEC','JAN','FEB','MAR','APR','MAY']

climperiod=30
if climperiod==30:
    startyear=0
else:
    startyear=0
typecast = np.array([0,1,2,3])

In [61]:
# To prevent the model from using things it shouldn't be
if (class_type != 'CPC') and (class_type != 'CHACE'):
    raise Exception('Choose classification set the model will use')


if class_type=='CPC':
    with open(f'{BASE_DIR}/rawrefinedcpc.pkl', "rb") as file:
        classset = pickle.load(file)
elif class_type=='CHACE':
    with open(f'{BASE_DIR}/rawrefinedchace.pkl', "rb") as file:
        classset = pickle.load(file)


print(f'Classification Approach: {class_type}')

# Structured as {month, features, parameters, H-Skill, and shorter climatology H-Skill}, for December to June

ENSO Phase: None
Classification Approach: CPC


# Various Useful Functions

In [62]:
def calculate_mode_params(param_list):
    """Finds the most frequent hyperparameter set from a list of dictionaries."""
    keys = param_list[0].keys()
    return {
        k: Counter([d[k] for d in param_list]).most_common(1)[0][0]
        for k in keys
    }

def calculate_likelihood(probabilities):
    """Calculates the geometric mean (likelihood) of a probability array."""
    return np.prod(probabilities) ** (1 / len(probabilities))

def align_class_probabilities(y_proba, model_classes, all_classes):
    """Maps model predicted probabilities to the full set of global classes."""
    full_probs = np.zeros(len(all_classes))
    for i, cls in enumerate(model_classes):
        full_probs[int(cls)] = y_proba[i]
    return full_probs

def get_climatology_bounds(year, base_year, clim_len=30, step=10):
    """Calculates the start and end years for a climatology reference block."""
    block_start = year - ((year - (base_year + clim_len)) % step)
    clim_start = block_start - clim_len
    clim_end = block_start - 1
    return clim_start, clim_end

def year_to_index(year, base_year=1951):
    return year - base_year

In [63]:
def breakdata(climate_modes,mode_data): 
    """
    Attached climate indices and months together

    Parameters:
        climate_modes: List of month names (e.g., ['JAN', 'FEB', ..., 'DEC']).
        ace_values: ACE (Accumulated Cyclonic Energy) as a 1D NumPy array.
        mode_data: Dictionary with climate mode names as keys and DataFrames with monthly data as values.

    Returns:
        Significant Features and Months
    """
    monthly_correlations = []
    namess_list = []
    for mode_name in mode_data:
        for month in climate_modes:
            df = mode_data[mode_name].copy()
            if month in df.columns:
                month_series = df[month]
            else:
                continue


            combined_name = f"{mode_name}_{month}"
            monthly_correlations.append(month_series)
            namess_list.append(combined_name)
    data = {name: series for name, series in zip(namess_list, monthly_correlations)}
    dz=pd.DataFrame(data)
    return dz

In [64]:
brokendata=breakdata(months_order,pure_data)

In [65]:
classset[0]

{'month': 'DEC',
 'features': ['amm_OCT',
  'amo_JUN',
  'ao_OCT',
  'epo_AUG',
  'ngst_SEP',
  'sgst_JUN',
  'mdrolr_NOV',
  'mdrslp_OCT',
  'pna_JUL',
  'pna_OCT',
  'tsa_JUL',
  'whwp_JUL',
  'whwp_AUG',
  'gomslp_SEP',
  'gomslp_OCT',
  'mdrt200_OCT',
  'mdru850_AUG',
  'mdrv200_JUN',
  'mdrv200_JUL',
  'mdrv200_AUG',
  'mdrv200_NOV',
  'mdrv850_JUL',
  'mdrv850_AUG',
  'mdrvws_JUN',
  'gomu850_JUL',
  'nino34_forecast_JUN',
  'nino34_forecast_JUL',
  'nino34_forecast_AUG'],
 'params': {'criterion': 'gini',
  'max_depth': 2,
  'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'n_estimators': 100},
 'testH': 0.05614161002963804,
 'neotestH': -0.06547516051028794}

# Call in Training Dataset to Forecast for New Year

In [66]:
year_val=2026
model_config = classset[0]
best_features = model_config['features']
best_params = model_config['params']
c_start, c_end = get_climatology_bounds(year_val, base_year=1951)
start_idx, end_idx = year_to_index(c_start), year_to_index(c_end) + 1
X_train, y_train = features[best_features].iloc[start_idx:end_idx], actualscore[start_idx:end_idx]

# If You are Doing a No ENSO Parameter Forecast

In [67]:
lister=[]


for i in best_features:
    if i == 'amm_JAN':
        lister.append(1.58)
    elif i == 'amo_FEB':
        lister.append(0.63)
    elif i == 'ngst_FEB':
        lister.append(160)
    elif i == 'mdrslp_JAN':
        lister.append(1014.919)
    elif i == 'sgst_MAR':
        lister.append(102)
    elif i == 'nino34_forecast_JUN':
        lister.append(1.0)
    elif i == 'nino34_forecast_JUL':
        lister.append(1.2)
    elif i == 'nino34_forecast_AUG':
        lister.append(1.5)
    else:
        lister.append(brokendata[i].iloc[75])
X_test=pd.DataFrame(data=np.asarray([lister]).reshape(-1,1).T,columns=best_features)

# Get Your Test Features Together and Make the Prediction!

In [68]:
# Actual Model Output
rf_classifier = RandomForestClassifier(**best_params, class_weight='balanced',random_state=42)

rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)[0]
y_prob = rf_classifier.predict_proba(X_test)[0]
print(y_prob)

[0.29173419 0.09724515 0.21811454 0.39290613]


In [49]:
print(y_prob)

[0.25038918 0.10382366 0.27591033 0.36987682]


In [42]:
print(y_prob)

[0.30365204 0.09319047 0.26190254 0.34125495]
